In [1]:
import pandas as pd

from utils import enhance

In [2]:
data = pd.read_csv("./data/train.csv", index_col=0)

to_lag = ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]
to_drop = ["E7", "V10", "S3", "M1", "M13", "M14", "M6", "V9"]  #缺失值多，直接删除列

#处理bool列
for i in range(1, 10):
    data[f"D{i}"] = data[f"D{i}"] != 0
    data[f"D{i}"] = data[f"D{i}"].astype("category")

#处理收益率列
for i in to_lag:
    data[f"lag_{i}"] = data[i].shift(1)

data["target"] = (data["forward_returns"] - data["risk_free_rate"]) > 0  #使用指数收益率是否大于无风险利率作为预测值
data = data.drop(to_drop + to_lag, axis=1)
data

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V3,V4,V5,V6,V7,V8,lag_forward_returns,lag_risk_free_rate,lag_market_forward_excess_returns,target
date_id,,,,,,,,,,,,,,,,,,,,,
0,False,False,False,True,True,False,False,False,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,False,False,False,True,True,False,False,False,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.002421,0.000301,-0.003038,False
2,False,False,False,True,False,False,False,False,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.008495,0.000303,-0.009114,False
3,False,False,False,True,False,False,False,False,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.009624,0.000301,-0.010243,True
4,False,False,False,True,False,False,False,False,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.004662,0.000299,0.004046,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9016,False,False,False,True,False,False,False,False,False,1.493117,...,0.208995,0.484788,0.717308,0.677249,-0.327455,0.083995,0.010401,0.000152,0.009936,False
9017,False,False,False,True,False,False,False,False,False,1.490889,...,0.082011,0.482804,1.001028,0.596561,-0.372979,0.094246,-0.000015,0.000151,-0.000477,False
9018,False,False,False,True,False,True,False,False,False,1.488667,...,0.334656,0.486772,0.894502,0.656746,-0.282024,0.090608,-0.005199,0.000150,-0.005661,True


In [3]:
x = data.drop("target", axis=1)

windows = [63, 126]
spans = [63, 126]
dataset = enhance.Dataset(x)
dataset.add_primitives([
    enhance.RollingMin(windows), #只对数值型的列生效
    # enhance.RollingMax(windows),
    # enhance.RollingMean(windows),
    # enhance.RollingStd(windows),
    # enhance.RollingCountTrue(windows),
    # enhance.RollingCountAboveMean(windows),
    # enhance.RollingTrend(windows),
    # enhance.RollingMaxConsecutiveTrue(windows),
    # enhance.RollingMaxConsecutivePositives(windows),
    # enhance.RollingNumSinceLastTrue(windows),
    enhance.EwmMean(spans), #只对数值型的列生效
    # enhance.EwmStd(spans),
    enhance.Lag(), #对所以类型的列生效
    # enhance.Diff(),
    # enhance.PctChange(),
])
df = dataset.build()[-5400:]  #选择没有缺失值的部分
df

RollingMin处理的列: ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'lag_forward_returns', 'lag_risk_free_rate', 'lag_market_forward_excess_returns']
EwmMean处理的列: ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S4', 'S5', 

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V2_Lag_P1,V3_Lag_P1,V4_Lag_P1,V5_Lag_P1,V6_Lag_P1,V7_Lag_P1,V8_Lag_P1,lag_forward_returns_Lag_P1,lag_risk_free_rate_Lag_P1,lag_market_forward_excess_returns_Lag_P1
date_id,,,,,,,,,,,,,,,,,,,,,
3621,False,False,False,True,False,False,False,False,False,1.069307,...,0.158730,0.271164,0.248677,-0.883869,0.000661,-0.801625,0.000661,-0.010276,0.000041,-0.010626
3622,False,False,True,True,False,False,False,False,False,1.068143,...,0.152778,0.114418,0.252646,-0.070909,0.000661,-1.007954,0.000661,0.008454,0.000041,0.008105
3623,False,False,False,True,False,False,False,False,False,1.066981,...,0.171296,0.248677,0.326720,-0.221641,0.000661,-1.081144,0.000661,0.006378,0.000041,0.006029
3624,False,False,False,True,False,False,False,False,False,1.065821,...,0.160053,0.093915,0.316138,-0.011475,0.000661,-0.956609,0.000661,-0.004165,0.000040,-0.004513
3625,False,False,False,False,False,False,False,False,False,1.064664,...,0.151455,0.119048,0.341270,-0.172679,0.000661,-1.023332,0.000661,0.000455,0.000039,0.000108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9016,False,False,False,True,False,False,False,False,False,1.493117,...,0.570767,0.126984,0.445767,0.680419,0.564815,-0.007844,0.111111,0.005676,0.000153,0.005211
9017,False,False,False,True,False,False,False,False,False,1.490889,...,0.541005,0.208995,0.484788,0.717308,0.677249,-0.327455,0.083995,0.010401,0.000152,0.009936
9018,False,False,False,True,False,True,False,False,False,1.488667,...,0.507937,0.082011,0.482804,1.001028,0.596561,-0.372979,0.094246,-0.000015,0.000151,-0.000477


In [6]:
windows = [20]
spans = [20]
dataset = enhance.Dataset(x)
dataset.add_primitives([
    enhance.RollingMin(windows, ["*", "~I1", "~E*"]),  #选择除了D1和E开头的所有列
    enhance.EwmMean(spans, ["E1*", "~E13"]),  #选择除了E13的所有E1开头的列
])
df = dataset.build()[-5400:]  #选择没有缺失值的部分
df

RollingMin处理的列: ['I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M10', 'M11', 'M12', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'lag_forward_returns', 'lag_risk_free_rate', 'lag_market_forward_excess_returns']
EwmMean处理的列: ['E1', 'E10', 'E11', 'E12', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19']


,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,E1_Mean_E20,E10_Mean_E20,E11_Mean_E20,E12_Mean_E20,E14_Mean_E20,E15_Mean_E20,E16_Mean_E20,E17_Mean_E20,E18_Mean_E20,E19_Mean_E20
date_id,,,,,,,,,,,,,,,,,,,,,
3621,False,False,False,True,False,False,False,False,False,1.069307,...,1.080458,0.147159,0.046267,0.062981,0.004250,0.875993,1.666868,2.107651,2.336487,-0.569193
3622,False,False,True,True,False,False,False,False,False,1.068143,...,1.079285,0.147505,0.049073,0.062683,0.004254,0.881882,1.680766,2.104662,2.354455,-0.496830
3623,False,False,False,True,False,False,False,False,False,1.066981,...,1.078113,0.147850,0.051580,0.062382,0.004227,0.887242,1.693051,2.101575,2.370068,-0.465257
3624,False,False,False,True,False,False,False,False,False,1.065821,...,1.076943,0.148193,0.053816,0.062079,0.004171,0.892123,1.703876,2.098400,2.383554,-0.419888
3625,False,False,False,False,False,False,False,False,False,1.064664,...,1.075773,0.148535,0.055809,0.061772,0.004089,0.896570,1.713382,2.095149,2.395119,-0.416820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9016,False,False,False,True,False,False,False,False,False,1.493117,...,1.514922,0.177288,0.006260,0.006260,0.003748,0.925125,-0.330648,-0.518123,0.121907,-0.249819
9017,False,False,False,True,False,False,False,False,False,1.490889,...,1.512633,0.176969,0.005853,0.005853,0.003580,0.925454,-0.335532,-0.516349,0.118557,-0.279258
9018,False,False,False,True,False,True,False,False,False,1.488667,...,1.510351,0.176649,0.005453,0.005453,0.003396,0.925782,-0.339965,-0.514764,0.115477,-0.249425
